# Approach 1 — GAN Training

**Before running:** Runtime → Change runtime type → **T4 GPU**

### What this trains
`StyleEncoder + CharacterGenerator + Discriminator (AC-GAN)`:  
A reference handwriting image is encoded into a style vector.  
The generator takes (style vector + character embedding) and produces a handwriting image.  
The discriminator judges real/fake AND classifies the character (prevents mode collapse).

### Steps
1. Verify GPU
2. Mount Drive
3. Install deps
4. Clone repo (data is included)
5. Train GAN
6. Save checkpoint to Drive + download
7. Preview generated characters

### After training
Download `checkpoint.pt` and place it at:  
`approaches/01_GAN/checkpoints/checkpoint.pt`  
Then run locally: `cd approaches/01_GAN && python run.py`

In [1]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

CUDA available: False


In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Install dependencies
!pip install scipy scikit-image -q
print('Dependencies installed')

In [ ]:
# Cell 4: Clone repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

APPROACH_DIR = f'{REPO_PATH}/approaches/01_GAN'
print('Approach dir:', APPROACH_DIR)
print('Files:', os.listdir(APPROACH_DIR))

In [ ]:
# Cell 5: Configure training
EPOCHS = 200   # increase for better quality (200 recommended for publication)

print(f'Will train for {EPOCHS} epochs')
print(f'Data: {REPO_PATH}/Data/Writers_pngs')

In [ ]:
# Cell 6: Train
os.chdir(APPROACH_DIR)
!python run.py --train --epochs {EPOCHS}

In [ ]:
# Cell 7: Save checkpoint to Drive + download
import shutil
from google.colab import files

ckpt_src  = f'{APPROACH_DIR}/checkpoints/checkpoint.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints/01_GAN'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'checkpoint.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('\nDownload started.')
print('Place checkpoint.pt in:  approaches/01_GAN/checkpoints/')
print('Then run locally:        cd approaches/01_GAN && python run.py')

In [ ]:
# Cell 8: Preview generated characters
import sys
sys.path.insert(0, APPROACH_DIR)

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from dataset     import CharDataset, load_all_writers, CHAR_TO_LABEL
from encoder     import StyleEncoder
from generator   import CharacterGenerator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(f'{APPROACH_DIR}/checkpoints/checkpoint.pt', map_location=device)

enc = StyleEncoder().to(device);       enc.load_state_dict(ckpt['enc']);  enc.eval()
gen = CharacterGenerator().to(device); gen.load_state_dict(ckpt['gen']); gen.eval()

DATA_ROOT = f'{REPO_PATH}/Data/Writers_pngs'
ref_img   = CharDataset(load_all_writers(DATA_ROOT))[0][0].unsqueeze(0).to(device)

chars = 'abcdefghij'
fig, axes = plt.subplots(1, len(chars), figsize=(len(chars)*2, 2))
with torch.no_grad():
    style = enc(ref_img)
    for ax, ch in zip(axes, chars):
        idx = CHAR_TO_LABEL[ch]
        out = gen(style, torch.tensor([idx], device=device))
        arr = ((out.squeeze().cpu().numpy() + 1) / 2 * 255).clip(0, 255).astype('uint8')
        ax.imshow(arr, cmap='gray', vmin=0, vmax=255)
        ax.set_title(ch); ax.axis('off')
plt.suptitle(f'GAN output after {EPOCHS} epochs')
plt.tight_layout()
plt.show()